Q1. 학생 성적 보고서 생성기

(1) 코드

In [4]:
import csv
import json
import logging
from typing import List, Dict, Optional

# 로깅 설정 (INFO 레벨)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def make_report(csv_path: str, json_path: str) -> int:
    """CSV 성적 파일을 읽어 가중 평균과 등급을 계산하고 JSON으로 저장한다."""
    students: List[Dict] = []
    processed_count: int = 0

    try:
        # (a) 읽기: with 문과 encoding 매개변수 사용
        with open(csv_path, mode='r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            
            for row in reader:
                name: str = row['이름']
                sid: str = row['학번']
                
                # 빈 칸(결측치) 확인
                raw_mid = row['중간'].strip()
                raw_final = row['기말'].strip()
                raw_assign = row['과제'].strip()
                
                has_missing: bool = not (raw_mid and raw_final and raw_assign)
                
                avg: Optional[float] = None
                grade: Optional[str] = None
                scores_dict: Dict[str, Optional[int]] = {
                    "중간": int(raw_mid) if raw_mid else None,
                    "기말": int(raw_final) if raw_final else None,
                    "과제": int(raw_assign) if raw_assign else None
                }

                # (b) 계산: 점수가 모두 있는 경우만 가중 평균 산출
                if not has_missing:
                    # 가중 평균: 중간 30%, 기말 50%, 과제 20%
                    avg = (scores_dict["중간"] * 0.3) + \
                          (scores_dict["기말"] * 0.5) + \
                          (scores_dict["과제"] * 0.2)
                    
                    # 등급 판정
                    if avg >= 90: grade = "A"
                    elif avg >= 80: grade = "B"
                    elif avg >= 70: grade = "C"
                    else: grade = "F"
                    
                    logging.info(f"{name}: 평균 {avg:.1f}, 등급 {grade}")
                else:
                    logging.info(f"{name}: 결측값이 포함되어 성적을 산출할 수 없습니다.")

                students.append({
                    "이름": name,
                    "학번": sid,
                    "점수": scores_dict,
                    "평균": avg,
                    "등급": grade
                })
                processed_count += 1

        # (c) 쓰기: 한글 보존 및 들여쓰기 적용
        with open(json_path, mode='w', encoding='utf-8') as jf:
            json.dump(students, jf, ensure_ascii=False, indent=2)
            
        return processed_count

    # (d) 예외 처리와 로그
    except FileNotFoundError:
        logging.warning(f"파일이 없습니다: {csv_path}")
        return 0
    except UnicodeDecodeError:
        logging.error(f"인코딩 오류가 발생했습니다: {csv_path}")
        return 0

# 실행 예시
count = make_report("scores.csv", "report.json")
print(f"\n성공적으로 처리된 학생 수: {count}")

INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 결측값이 포함되어 성적을 산출할 수 없습니다.



성공적으로 처리된 학생 수: 4


(2) 실행 결과

INFO: 김언어: 평균 89.5, 등급 B
INFO: 이국문: 평균 84.4, 등급 B
INFO: 박영문: 평균 93.5, 등급 A
INFO: 최역사: 결측값이 있어 성적을 산출할 수 없습니다.

성공적으로 처리된 학생 수: 4

(3) 설명

접근 방식: csv.DictReader를 사용하여 데이터를 키-값 쌍으로 처리하고, 조건문을 통해 점수 필드 중 하나라도 비어있는 학생은 평균과 등급을 None으로 처리했다. 

인코딩 선택 근거: 한글 텍스트의 표준인 utf-8을 사용하여 다국어 호환성을 확보했으며, JSON 저장 시 ensure_ascii=False를 설정하여 한글이 그대로 보존되게 했다.

결측값 처리 전략: 결측값이 존재할 경우 수치 계산을 생략하고 null 값을 할당하여 데이터의 무결성을 유지하고 로그를 통해 해당 사실을 기록했다. 

Q2. 사용자 정의 예외와 자모 분류

(1) 코드

In [8]:
# (a) 사용자 정의 예외 클래스 정의
class InvalidJamoError(ValueError):
    """한글 자모 범위를 벗어난 문자가 입력되었을 때 발생하는 예외"""
    pass

# (b) 함수 classify_jamo 구현
def classify_jamo(c: str) -> str:
    """단일 한글 자모를 받아 자음/모음을 분류한다."""
    # 1. 타입 검사 (TypeError)
    if not isinstance(c, str):
        raise TypeError(f"문자열이 아닙니다: {type(c)}")
    
    # 2. 길이 검사 (ValueError)
    # 빈 문자열 "" 이나 "AB"는 여기서 걸러집니다.
    if len(c) != 1:
        raise ValueError(f"한 글자가 아닙니다: '{c}'")

    code: int = ord(c)
    
    # 3. 자음/모음 범위 확인 (유니코드 U+3131~314E, U+314F~3163)
    if 0x3131 <= code <= 0x314E:
        return "자음"
    elif 0x314F <= code <= 0x3163:
        return "모음"
    else:
        # 길이가 1이지만 자모 범위가 아닌 경우 ("가", " ", "!" 등)[cite: 1]
        raise InvalidJamoError(f"유효한 한글 자모가 아님: {c}")

# (c) 입력 리스트 처리
# 마지막 공백("")을 포함한 문제지의 모든 케이스 반영[cite: 1]
inputs = ["ㄱ", "ㅏ", "가", " ", "AB", 5, "!", "ㅎ", "ㅣ", ""]

for item in inputs:
    try:
        res: str = classify_jamo(item)
        print(f"'{item}': {res}")
    except TypeError as e:
        print(f"[TypeError] {e}")
    except InvalidJamoError as e:
        # InvalidJamoError는 ValueError의 자식이므로 먼저 잡아야 함[cite: 1]
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")

'ㄱ': 자음
'ㅏ': 모음
[InvalidJamoError] 유효한 한글 자모가 아님: 가
[InvalidJamoError] 유효한 한글 자모가 아님:  
[ValueError] 한 글자가 아닙니다: 'AB'
[TypeError] 문자열이 아닙니다: <class 'int'>
[InvalidJamoError] 유효한 한글 자모가 아님: !
'ㅎ': 자음
'ㅣ': 모음
[ValueError] 한 글자가 아닙니다: ''


(2) 실행 결과

'ㄱ': 자음
'ㅏ': 모음
[InvalidJamoError] 유효한 한글 자모가 아님: 가
[InvalidJamoError] 유효한 한글 자모가 아님:  
[ValueError] 한 글자가 아닙니다: 'AB'
[TypeError] 문자열이 아닙니다: <class 'int'>
[InvalidJamoError] 유효한 한글 자모가 아님: !
'ㅎ': 자음
'ㅣ': 모음
[ValueError] 한 글자가 아닙니다: ''

(3) 설명

ValueError 상속 이유: 입력값의 '자료형'은 올바르지만, '내용'이 특정 범위(한글 자모)를 벗어나는 논리적 오류를 나타내므로 Exception보다 구체적인 ValueError를 상속하는 것이 적절하다. 

접근 방식: ord()를 통해 각 문자의 유니코드 값을 확인하고, 문제에서 지정된 자음(U+3131-U+314E) 및 모음(U+314F-U+3163) 영역과 비교했다. 

예외 처리 전략: 자식 예외인 InvalidJamoError를 부모인 ValueError보다 상단에 배치하여 예외를 정확히 분류하고, 프로그램이 중단되지 않도록 반복문 내에서 개별적으로 catch했다. 